# SA-DMAE Segmentation Fine-tuning

BraTS 2021 종양 segmentation (WT / TC / ET)  
SA-DMAE pre-trained encoder (frozen) → segmentation head fine-tuning

- 이미지: .pt에서 로드 (빠름)
- Encoder 고정, decoder만 학습 (pre-trained 보존)
- Dice-only loss (class imbalance 강건)

> 런타임 → 런타임 유형 변경 → **GPU (T4)** 먼저 설정!

In [ ]:
# ── Cell 1: GPU 확인 & Drive 마운트 ──────────────────────────────────────────
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM           :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: 코드 & 패키지 ────────────────────────────────────────────────────
import os

REPO_DIR = '/content/SA-DMAE'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/whkim4338/SA-DMAE.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install timm nibabel tensorboard tqdm -q
print('완료')

In [ ]:
# ── Cell 3: 경로 설정 (여기만 수정) ─────────────────────────────────────────

# 전처리된 BraTS .pt 파일 디렉터리
BRATS_PT     = '/content/drive/MyDrive/SA_DMAE_slices/brats'   # ← 수정

# BraTS2021 NIfTI 원본 (seg 파일 추출용)
BRATS_NIFTI  = '/content/drive/MyDrive/BraTS2021'              # ← 수정

# SA-DMAE pre-trained 체크포인트
SA_DMAE_CKPT = '/content/drive/MyDrive/SA_DMAE_output/checkpoint-best.pth'

# 출력 디렉터리
OUTPUT_SA    = '/content/drive/MyDrive/SA_DMAE_seg/sa_dmae'

# 학습 설정
EPOCHS      = 50
BATCH_SIZE  = 16     # encoder frozen → VRAM 여유 있어서 16 가능
LR          = 1e-3   # decoder만 학습하므로 높은 LR 사용
VAL_RATIO   = 0.2
PATIENCE    = 15
SAVE_EVERY  = 10

import os
os.makedirs(OUTPUT_SA, exist_ok=True)

from pathlib import Path
pt_count = len(list(Path(BRATS_PT).glob('*.pt')))
print(f'BraTS .pt   : {pt_count}개')
print(f'SA-DMAE ckpt: {Path(SA_DMAE_CKPT).exists()}')

In [ ]:
# ── Cell 4: TensorBoard ───────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_SA}

In [ ]:
# ── Cell 5: SA-DMAE Segmentation Fine-tuning ──────────────────────────────────
# freeze_encoder: encoder 고정, decoder만 학습 (기본값)
# dice_only     : Dice loss만 사용, class imbalance 강건 (기본값)
!python main_finetune_seg.py \
    --pt_dir      {BRATS_PT} \
    --nifti_dir   {BRATS_NIFTI} \
    --resume      {SA_DMAE_CKPT} \
    --n_slices    3 \
    --axial_depth 4 \
    --epochs      {EPOCHS} \
    --batch_size  {BATCH_SIZE} \
    --lr          {LR} \
    --val_ratio   {VAL_RATIO} \
    --patience    {PATIENCE} \
    --save_every  {SAVE_EVERY} \
    --output_dir  {OUTPUT_SA} \
    --log_dir     {OUTPUT_SA} \
    --device      cuda \
    --num_workers 2

In [ ]:
# ── Cell 5-R: 이어서 학습 (세션 끊겼을 때) ───────────────────────────────────
import glob, os
ckpts = sorted(
    [f for f in glob.glob(f'{OUTPUT_SA}/checkpoint-*.pth') if 'best' not in f and 'final' not in f],
    key=os.path.getmtime
)
if ckpts:
    latest = ckpts[-1]
    print(f'이어서 학습: {latest}')
    !python main_finetune_seg.py \
        --pt_dir      {BRATS_PT} \
        --nifti_dir   {BRATS_NIFTI} \
        --resume      {SA_DMAE_CKPT} \
        --resume_seg  {latest} \
        --n_slices    3 \
        --axial_depth 4 \
        --epochs      {EPOCHS} \
        --batch_size  {BATCH_SIZE} \
        --lr          {LR} \
        --val_ratio   {VAL_RATIO} \
        --patience    {PATIENCE} \
        --save_every  {SAVE_EVERY} \
        --output_dir  {OUTPUT_SA} \
        --log_dir     {OUTPUT_SA} \
        --device      cuda \
        --num_workers 2
else:
    print('저장된 체크포인트 없음. Cell 5를 먼저 실행하세요.')

In [ ]:
# ── Cell 6: 결과 확인 ─────────────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

logs = [json.loads(l) for l in open(f'{OUTPUT_SA}/log_seg.txt')]
best = max(logs, key=lambda d: d['dice_mean'])

print('=' * 50)
print('SA-DMAE Segmentation 결과 (Best Val Dice)')
print('=' * 50)
print(f'  WT  (Whole Tumor)    : {best["dice_wt"]:.4f}')
print(f'  TC  (Tumor Core)     : {best["dice_tc"]:.4f}')
print(f'  ET  (Enhancing Tumor): {best["dice_et"]:.4f}')
print(f'  Mean Dice            : {best["dice_mean"]:.4f}')
print(f'  Best epoch           : {best["epoch"]+1}')
print('=' * 50)

epochs    = [d['epoch'] + 1  for d in logs]
dice_wt   = [d['dice_wt']    for d in logs]
dice_tc   = [d['dice_tc']    for d in logs]
dice_et   = [d['dice_et']    for d in logs]
dice_mean = [d['dice_mean']  for d in logs]
val_loss  = [d['val_loss']   for d in logs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('SA-DMAE Segmentation Fine-tuning', fontsize=12)

axes[0].plot(epochs, val_loss, color='tab:red', linewidth=2)
axes[0].set_title('Val Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, dice_wt,   label='WT',   linewidth=2)
axes[1].plot(epochs, dice_tc,   label='TC',   linewidth=2)
axes[1].plot(epochs, dice_et,   label='ET',   linewidth=2)
axes[1].plot(epochs, dice_mean, label='Mean', linewidth=2, linestyle='--', color='black')
axes[1].set_title('Val Dice Score'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_SA}/seg_result.png', dpi=120)
plt.show()

with open(f'{OUTPUT_SA}/seg_metrics.json', 'w') as f:
    json.dump({'sa_dmae': {'dice_wt': best['dice_wt'], 'dice_tc': best['dice_tc'],
                           'dice_et': best['dice_et'], 'dice_mean': best['dice_mean']}}, f, indent=2)
print('저장 완료: seg_metrics.json')